# Step 2 — Land opportunity cost accounting

This notebook estimates land opportunity costs by combining cropland and pasture area, allocating a share of cropland to livestock feed, and monetizing foregone carbon recovery under low, mid, and high SCC scenarios.

## Inputs and outputs

**Inputs**

- `../data/raw/FAOSTAT_cropland_2000-2022.csv`
- `../data/raw/FAOSTAT_land_meadows_pasture_2000-2022.csv`

**Outputs**

- `../data/outputs/land_opportunity_cost_country_year.csv`
- `../data/outputs/land_opportunity_cost_global_year.csv`

Place the notebook in `code/` and keep the raw and output files in parallel `data/raw` and `data/outputs` folders.

## Notebook workflow

This notebook is organized into short executable sections so that each transformation step is visible in GitHub and easy for reviewers to follow.

## Analysis step

In [ ]:
import pandas as pd
import numpy as np

## FILE PATHS

In [ ]:
CROPLAND_PATH = "./FAOSTAT_cropland_2000-2022.csv"
PASTURE_PATH  = "./FAOSTAT_land_meadows_pasture_2000-2022.csv"
OUT_COUNTRY   = "./land_opportunity_cost_country_year.csv"
OUT_GLOBAL    = "./land_opportunity_cost_global_year.csv"

## PARAMETERS

In [ ]:
FEED_SHARE = 0.33
RECOVERY_YEARS = 30

RECOVERABLE_STOCK_TCO2E_PER_HA = {
    "low": 75.0,
    "mid": 150.0,
    "high": 250.0
}

SCC_VALUES = {
    "low": 50,
    "mid": 100,
    "high": 200
}

KEYS = ["Area", "Area Code (M49)", "Year"]

## HELPER FUNCTION: Convert to hectares

In [ ]:
def to_hectares(df):
    df = df.copy()
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    df = df.dropna(subset=["Value"])
    
    unit = df["Unit"].astype(str).str.lower()
    
    df["area_ha"] = df["Value"]
    
    # Convert 1000 ha → ha
    mask = unit.str.contains("1000") & unit.str.contains("ha")
    df.loc[mask, "area_ha"] = df.loc[mask, "Value"] * 1000.0
    
    return df

## LOAD DATA

In [ ]:
cropland = pd.read_csv(CROPLAND_PATH)
pasture  = pd.read_csv(PASTURE_PATH)

# Keep only area rows if needed
if "Element" in cropland.columns:
    cropland = cropland[cropland["Element"].str.contains("Area", case=False, na=False)]

if "Element" in pasture.columns:
    pasture = pasture[pasture["Element"].str.contains("Area", case=False, na=False)]

# Convert to hectares
cropland = to_hectares(cropland)
pasture  = to_hectares(pasture)

## AGGREGATE COUNTRY-YEAR LAND

In [ ]:
cropland_cty = (
    cropland.groupby(KEYS, as_index=False)
            .agg(cropland_ha=("area_ha", "sum"))
)

pasture_cty = (
    pasture.groupby(KEYS, as_index=False)
           .agg(pasture_ha=("area_ha", "sum"))
)

land = cropland_cty.merge(pasture_cty, on=KEYS, how="inner")

# Feed cropland allocation
land["feed_cropland_ha"] = land["cropland_ha"] * FEED_SHARE

# Total livestock land
land["livestock_land_ha"] = land["pasture_ha"] + land["feed_cropland_ha"]

## CREATE SCENARIO TABLE

In [ ]:
params = pd.DataFrame([
    {
        "scenario": s,
        "SCC_usd_per_tCO2e": SCC_VALUES[s],
        "recoverable_stock_tco2e_per_ha": RECOVERABLE_STOCK_TCO2E_PER_HA[s],
        "recovery_years": RECOVERY_YEARS
    }
    for s in ["low", "mid", "high"]
])

params["annualized_seq_tco2e_per_ha_yr"] = (
    params["recoverable_stock_tco2e_per_ha"] / params["recovery_years"]
)

## CROSS JOIN COUNTRY-YEAR × SCENARIO

In [ ]:
land["key"] = 1
params["key"] = 1

land_full = land.merge(params, on="key").drop(columns="key")

## CALCULATE FOREGONE SEQUESTRATION

In [ ]:
land_full["foregone_co2e_tonnes"] = (
    land_full["livestock_land_ha"] *
    land_full["annualized_seq_tco2e_per_ha_yr"]
)

# Monetize
land_full["land_opportunity_cost_usd"] = (
    land_full["foregone_co2e_tonnes"] *
    land_full["SCC_usd_per_tCO2e"]
)

## SAVE COUNTRY-YEAR FILE

In [ ]:
land_full.to_csv(OUT_COUNTRY, index=False)

print("Saved country-year file:", OUT_COUNTRY)
print(land_full.head())

## CREATE GLOBAL-YEAR AGGREGATION

In [ ]:
land_global = (
    land_full.groupby(["scenario", "Year"], as_index=False)
    .agg(
        cropland_ha=("cropland_ha", "sum"),
        pasture_ha=("pasture_ha", "sum"),
        feed_cropland_ha=("feed_cropland_ha", "sum"),
        livestock_land_ha=("livestock_land_ha", "sum"),
        foregone_co2e_tonnes=("foregone_co2e_tonnes", "sum"),
        land_opportunity_cost_usd=("land_opportunity_cost_usd", "sum")
    )
)

land_global.to_csv(OUT_GLOBAL, index=False)

print("Saved global-year file:", OUT_GLOBAL)
print(land_global.head())